# This code will be used to read the data coming from the "FGCZ"

This code is function oriented but I will try to make the transition to Object Oriented Programming OOP)

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

## Extracting information from the "All" dataset, reorganizing and refeactoring

Taking data from the DE_Groups_vs_controls.xlsx sheet: diff_ex_analysis_wide

1. First I am only taking the FC information. 

In [68]:
all_raw_df = pd.read_excel("") # 'Experiment/1_hTERT_HME1/Data/All/DE_Groups_vs_Controls.xlsx', sheet_name='diff_exp_analysis_wide')

keep_columns = ['protein_Id', 'description', 'site', 'diff.full_vs_starve',
                'diff.EGF1_vs_starve', 'diff.EGF2_vs_starve', 'diff.EGF5_vs_starve', 'diff.EGF10_vs_starve', 'diff.EGF90_vs_starve',
                'diff.INS1_vs_starve', 'diff.INS2_vs_starve', 'diff.INS5_vs_starve', 'diff.INS10_vs_starve', 'diff.INS90_vs_starve',
                'diff.EGFnINS1_vs_starve', 'diff.EGFnINS2_vs_starve', 'diff.EGFnINS5_vs_starve', 'diff.EGFnINS10_vs_starve', 'diff.EGFnINS90_vs_starve' ]

clean_df = all_raw_df[keep_columns].copy()

clean_df.rename(columns={'protein_Id' : 'protein_ID',
                         'description' : 'prot_name',
                         'diff.EGF1_vs_starve' : 'EGF_1',
                         'diff.EGF2_vs_starve' : 'EGF_2',
                         'diff.EGF5_vs_starve' : 'EGF_5',
                         'diff.EGF10_vs_starve' : 'EGF_10',
                         'diff.EGF90_vs_starve' : 'EGF_90',
                         'diff.INS1_vs_starve' : 'INS_1',
                         'diff.INS2_vs_starve' : 'INS_2',
                         'diff.INS5_vs_starve' : 'INS_5',
                         'diff.INS10_vs_starve' : 'INS_10',
                         'diff.INS90_vs_starve' : 'INS_90',
                         'diff.EGFnINS1_vs_starve' : 'EGFnINS_1',
                         'diff.EGFnINS2_vs_starve' : 'EGFnINS_2',
                         'diff.EGFnINS5_vs_starve' : 'EGFnINS_5',
                         'diff.EGFnINS10_vs_starve' : 'EGFnINS_10',
                         'diff.EGFnINS90_vs_starve' : 'EGFnINS_90',
                         'diff.full_vs_starve' : 'EGF_full' #EGF_full
                       },inplace=True)

clean_df.insert(4,'EGF_starve',0) 
clean_df.insert(10,'INS_full', clean_df.EGF_full)
clean_df.insert(11,'INS_starve',0)   
clean_df.insert(17,'EGFnINS_full', clean_df.EGF_full) 
clean_df.insert(18,'EGFnINS_starve',0)   

# To this point the dataframe only has the FC information, and the conditions are in continuous way
# clean_df.to_excel("Experiment/1_hTERT_HME1/Data/Processed/full_starve_continuous_FC.xlsx", index = False)


# Put the data grouped by conditions
# grouped_columns = ['protein_ID', 'prot_name', 'site', 'condition', 'full', 'starve', 'min1', 'min2', 'min5', 'min10', 'min90']
# grouped_df = pd.DataFrame(columns = grouped_columns)
# 
# for row in clean_df.itertuples():
#     for i in [0,5,10]:
#         if i == 0:
#             condition = 'EGF'
#         elif i == 5:
#             condition = 'INS'
#         elif i == 10:
#             condition = 'EGFnINS'
# 
#         new_row = {'protein_ID' : row[1],
#                    'prot_name' : row[2],
#                    'site' : row[3],
#                    'condition' : condition,
#                    'full' : row[4],
#                    'starve' : row[5],
#                    'min1' : row[6+i],
#                    'min2' : row[7+i],
#                    'min5' : row[8+i],
#                    'min10' : row[9+i],
#                    'min90' : row[10+i]}
# 
#         grouped_df = grouped_df._append(new_row, ignore_index=True)
# 
# grouped_df.to_excel("Experiment/1_hTERT_HME1/Data/Processed/full_starve_grouped_FC.xlsx", index = False)


FileNotFoundError: [Errno 2] No such file or directory: ''

## Add the standard deviation (SD), the number of replicates

For the SD i am going to take the values from [Experiment/1_hTERT_HME1/Data/All/DE_Groups_vs_Controls.xlsx](Experiment/1_hTERT_HME1/Data/All/DE_Groups_vs_Controls.xlsx) the sheet **stats_normalized_wide**

From here I am onl taking the "sd", however I could also select the "var" and "mean"



In [67]:
path_to_add = "" # "Experiment/1_hTERT_HME1/Data/Processed/full_starve_continuous_FC.xlsx"
path_to_stats = "Experiment/1_hTERT_HME1/Data/All/DE_Groups_vs_Controls.xlsx"
path_to_save = path_to_add.replace('.xlsx','_nrep_SD.xlsx')

df_to_add = pd.read_excel(path_to_add)
# df_to_add = clean_df.copy()
stats_df = pd.read_excel(path_to_stats, sheet_name='stats_normalized_wide')

keep_stats2 = ['protein_Id', 'site', 'not_na_full', 'sd_All',
              'sd_full_EGF', 'sd_starve_EGF', 'sd_EGF1', 'sd_EGF2', 'sd_EGF5', 'sd_EGF10', 'sd_EGF90',
              'sd_full_INS', 'sd_starve_INS', 'sd_INS1', 'sd_INS2', 'sd_INS5',  'sd_INS10', 'sd_INS90',
              'sd_full_EGFnINS', 'sd_starve_EGFnINS', 'sd_EGFnINS1', 'sd_EGFnINS2', 'sd_EGFnINS5', 'sd_EGFnINS10', 'sd_EGFnINS90']

df_to_add[keep_stats2[2:]] = 0
df_to_add.rename(columns={'not_na_full' : 'n_replicates'},inplace=True)

for row in df_to_add.itertuples(): # 47 minutes to run (around 2 mins for each condition)
    id = row[3]
    df_to_add.n_replicates.loc[df_to_add['site'] == id ] = stats_df.not_na_starve.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_All.loc[df_to_add['site'] == id ] = stats_df.sd_All.loc[stats_df['site'] == id].iloc[0]

    df_to_add.sd_full_EGF.loc[df_to_add['site'] == id ] = stats_df.sd_full.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_starve_EGF.loc[df_to_add['site'] == id ] = stats_df.sd_starve.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_EGF1.loc[df_to_add['site'] == id ] = stats_df.sd_EGF1.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_EGF2.loc[df_to_add['site'] == id ] = stats_df.sd_EGF2.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_EGF5.loc[df_to_add['site'] == id ] = stats_df.sd_EGF5.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_EGF10.loc[df_to_add['site'] == id ] = stats_df.sd_EGF10.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_EGF90.loc[df_to_add['site'] == id ] = stats_df.sd_EGF90.loc[stats_df['site'] == id].iloc[0]

    df_to_add.sd_full_INS.loc[df_to_add['site'] == id ] = stats_df.sd_full.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_starve_INS.loc[df_to_add['site'] == id ] = stats_df.sd_starve.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_INS1.loc[df_to_add['site'] == id ] = stats_df.sd_INS1.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_INS2.loc[df_to_add['site'] == id ] = stats_df.sd_INS2.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_INS5.loc[df_to_add['site'] == id ] = stats_df.sd_INS5.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_INS10.loc[df_to_add['site'] == id ] = stats_df.sd_INS10.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_INS90.loc[df_to_add['site'] == id ] = stats_df.sd_INS90.loc[stats_df['site'] == id].iloc[0]

    df_to_add.sd_full_EGFnINS.loc[df_to_add['site'] == id ] = stats_df.sd_full.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_starve_EGFnINS.loc[df_to_add['site'] == id ] = stats_df.sd_starve.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_EGFnINS1.loc[df_to_add['site'] == id ] = stats_df.sd_EGFnINS1.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_EGFnINS2.loc[df_to_add['site'] == id ] = stats_df.sd_EGFnINS2.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_EGFnINS5.loc[df_to_add['site'] == id ] = stats_df.sd_EGFnINS5.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_EGFnINS10.loc[df_to_add['site'] == id ] = stats_df.sd_EGFnINS10.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_EGFnINS90.loc[df_to_add['site'] == id ] = stats_df.sd_EGFnINS90.loc[stats_df['site'] == id].iloc[0]

df_to_add.to_excel("Experiment/1_hTERT_HME1/Data/Processed/full_starve_continuous_FC_nrep_SD.xlsx", index = False)

FileNotFoundError: [Errno 2] No such file or directory: ''

## Filter by Log2FC values, number of replicates and/or number of phosphorylations identified in the peptide

In [27]:
def check_max_amplitude(row):
    '''Returnd the maximun amplitude of the phosphosite'''
    c = 0
    for element in row[:24]:
        if type(element) == float:
            FC_list = row[c:24]
            max_FC = max(FC_list)
            break
        c += 1
    return max_FC

def check_min_amplitude(row):
    '''Returnd the maximun amplitude of the phosphosite'''
    c = 0
    for element in row[:24]:
        if type(element) == float:
            FC_list = row[c:24]
            min_FC = min(FC_list)
            break
        c += 1
    return min_FC

def check_all_localized(phosphorylation_site):
    '''Returns true or false deppending if all phosphorylations sites are localized '''
    phosphopeptide_list = re.split("_|~", phosphorylation_site)
    n_ph = phosphopeptide_list[3]
    found_ph = phosphopeptide_list[4]
    if len(found_ph) == 0:
         return False
    else:
        if n_ph == found_ph:
            return True
        else:
            return False

In [59]:
path = "" #"Experiment/1_hTERT_HME1/Data/Processed/full_starve_continuous_FC_nrep_SD.xlsx"
log2FC_threshold = 0 # Will keep sites with a log2FC grater than the threshold found at least in one of the time points
n_replicates = 0 # Will remove all site with the same or less number of replicates
all_phosphosites_found = False # If this is activated will keep only the sites where we know where are the phosphorylations happening
to_save = path.replace(".xlsx", f"_log2FC{log2FC_threshold}_nRep{n_replicates}_allPhosphoFound_{str(all_phosphosites_found)}.xlsx")

df = pd.read_excel(path)
for row in df.itertuples():
    df_index = row[0] 
    # Drop by log2FC threshold
    minimun = check_min_amplitude(row)
    maximun = check_max_amplitude(row)
    if maximun >= log2FC_threshold or -1*minimun >= log2FC_threshold:
        pass
    else:
        df.drop(index=df_index, inplace=True)
        continue       
    # Drop by replicates
    replicates = row[25]
    if replicates <= n_replicates:
        df.drop(index=df_index, inplace=True)
        continue   
    # Drop sites that don't have all phosphorylation sites localized
    if all_phosphosites_found == True:
        site = row[3]
        all_localized = check_all_localized(site)
        if all_localized == False:
            df.drop(index=df_index, inplace=True)
            continue     
# Saving file
df.to_excel(to_save, index=False)

FileNotFoundError: [Errno 2] No such file or directory: ''

## Expressing the curve behaviour as qualitative change

In [66]:
path = "" #"Experiment/1_hTERT_HME1/Data/Processed/full_starve_continuous_FC_nrep_SD.xlsx"
error_allowed = 0.1 # Percentage error allowed to consider the step as "stay"
to_save = path.replace(".xlsx", "_qualitative_description.xlsx")

df = pd.read_excel(path)
# Adding colums for each of the conditions behaviour
df["EGF_qualitative"] = False
df["INS_qualitative"] = False
df["EGFnINS_qualitative"] = False
qualitative_list_names = ["EGF_qualitative", "INS_qualitative", "EGFnINS_qualitative"]
for row in df.itertuples():
    c = 4 # Time point data start in colum 4
    replicates = row[24] # number of replicates the phosphosite has
    qualitative_list = [] 
    string = ""
    #Check the maximum amplitude and calculate the error allowed based on that
    minimun = check_min_amplitude(row)
    maximun = check_max_amplitude(row)
    if minimun < 0: # if minimum is negative, transform it to positive
        minimun = minimun * -1    
    value_error_allowed = (max(maximun, minimun)*error_allowed)
    #Qualitative analysis (There are 6 possible steps)
    for i in range(21):
        # Skipping the transition from one condition to the other (from EGF_90 to full_INS, and to INS_90 to full_EGFnINS)
        if c+i in [10,17,24]: 
            qualitative_list.append(string[:-1]) #remove the last "_"
            string = ""
            continue
            # If I wanted to skip more steps I could include these values for the skipping part of the algorithm
            # if c+i in [4, 10, 9,         #full to cero, 90 to next full, 10 to 90
            #           11, 17, 16,       #full to cero, 90 to next full, 10 to 90
            #           18, 24, 23]:   #full to cero, 90 to next full, 10 to 90
        # Qualitative step        
        if (row[c+i] < row[c+i+1]-value_error_allowed):
            string = string+"up_"
        elif (row[c+i] > row[c+i+1]+value_error_allowed):
            string = string+"down_"
        else:
            string = string+"stay_"
    # Add the information to the dataframe
    c = 0 # This is for the tree conditions: EGF, INS and EGFnINS
    for element in qualitative_list_names:
        df.at[row[0], element] = qualitative_list[c]
        c += 1
# Saving file
df.to_excel(to_save, index=False)

FileNotFoundError: [Errno 2] No such file or directory: ''

here i am missing 
    from the Data_filtration file
        the checking string networks and transcription factors
    from the Ochoa_filtering 
        filter based on ochoa functional score
    from data_extractino
        analyse information from the database
        